In [21]:
# Código que venía en el notebook Kaggle, eliminados comentarios

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('../data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

../data/solicitudes_test.csv
../data/solicitudes_train.csv
../data/comportamiento_bancario.csv


In [22]:
from sklearn.ensemble import RandomForestClassifier

# Carga de Datos
base = "../data/"
train = pd.read_csv(base + "solicitudes_train.csv")
test = pd.read_csv(base + "solicitudes_test.csv")
behaviour = pd.read_csv(base + "comportamiento_bancario.csv")

In [23]:
# Agrupado de Comportamientos
aggregated_behaviour = behaviour.groupby("ID_Usuario").agg(
    avg_monthly_balance=("Saldo_Promedio_Mensual", "mean"),
    max_days_past_due=("Dias_Atraso_Otras_Deudas", "max")
).reset_index()

# JOINS de dataset con lo agrupado
train_merged = train.merge(aggregated_behaviour, on="ID_Usuario", how="left").fillna(0)
test_merged = test.merge(aggregated_behaviour, on="ID_Usuario", how="left").fillna(0)

In [24]:
display(train_merged.head())

display(test_merged.head())

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final,avg_monthly_balance,max_days_past_due
0,1,12698,7900,0.2333,4461,Consumo,2025-04,1,3567.364783,25
1,2,20744,17400,0.2507,3889,Vivienda,2025-12,1,6070.002273,56
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06,0,168.037273,57
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07,1,7564.528696,53
4,5,24719,6600,0.2314,3357,Otros,2025-01,1,3504.671304,32


,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,avg_monthly_balance,max_days_past_due
0,32417,27074,2300,0.2771,2596,Consumo,2026-01,2104.447500,50
1,32418,28172,10900,0.3291,6905,Consumo,2026-01,21749.880455,43
2,32419,27547,1300,0.2485,3707,Educacion,2026-01,1211.650435,67
3,32420,27689,1600,0.2699,3314,Consumo,2026-01,7926.885455,61
4,32421,26219,900,0.2317,3870,Educacion,2026-01,6129.496087,53


In [25]:
# Columnas categóricas (One-Hot encode para que funcionen con árboles) 
train_encoded = pd.get_dummies(train_merged, columns=["Motivo_Prestamo"], drop_first=True)
test_encoded = pd.get_dummies(test_merged, columns=["Motivo_Prestamo"], drop_first=True)

# Estructuramos test_encoded al formato del train
test_encoded = test_encoded.reindex(columns=train_encoded.columns.drop("Estado_Final"), fill_value=0)

In [26]:
# Columnas de relleno, no útiles para entrenamiento
drop_columns = ["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]

# Separación de X (variables predictorias) e Y (objetivo)
X_train = train_encoded.drop(columns=drop_columns + ["Estado_Final"])
y_train = train_encoded["Estado_Final"]
X_test = test_encoded.drop(columns=drop_columns)

In [27]:
# Entrenamiento y estimación

# Primera Entrega (top 1)
rf_model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42)

# Segunda Entrega (top 3)
# rf_model = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42)

# Tercera Entrega (top 2)
# rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

rf_model.fit(X_train, y_train)

test["repayment_probability"] = rf_model.predict_proba(X_test)[:, 1]

In [28]:
gain = test["Monto_Solicitado"] * test["Tasa_Interes"]
loss = test["Monto_Solicitado"]

# Ganancia Esperada
# SI Resultado > 0: La operación es rentable en promedio.
# SINO La tasa de interés no compensa el riesgo de impago (prestar genera pérdidas a largo plazo)
test["expected_profit"] = (test["repayment_probability"] * gain) - ((1 - test["repayment_probability"]) * loss)

# Retorno de inversión (punto óptimo entre seguridad y ganancia)
# Evita que todos los préstamos sean pocos de mucho valor, diversifica
test["expected_roi"] = test["expected_profit"] / test["Monto_Solicitado"]

In [29]:
# Filtra solo los que tienen esperanza matemática positiva
candidates = test[test["expected_profit"] > 0].copy()

# Ordena ESTRICTAMENTE por mayor retorno de inversion
candidates = candidates.sort_values(by="expected_roi", ascending=False)

In [30]:
# Asigna el presupuesto a los más seguros hasta agotar los 2.000.000 USD
candidates["cumulative_amount"] = candidates["Monto_Solicitado"].cumsum()

BUDGET_LIMIT = 2000000
# Obtiene los préstamos hasta llegar al acumulado que no excede el capital inicial
approved_loans = candidates[candidates["cumulative_amount"] <= BUDGET_LIMIT]
# Obtiene los ids de préstamos
approved_ids = approved_loans["ID_Solicitud"].tolist()

In [35]:
# Colocar estados de préstamos en test
test["Decision_Aprobacion"] = 0
test.loc[test["ID_Solicitud"].isin(approved_ids), "Decision_Aprobacion"] = 1

# Generación del CSV 
submission = test[["ID_Solicitud", "Decision_Aprobacion"]]
submission.to_csv("submits/mas_probable_cobro_ROI.csv", index=False)

In [32]:
# Métricas de control
total_allocated = test[test["Decision_Aprobacion"] == 1]["Monto_Solicitado"].sum()
min_prob = approved_loans["repayment_probability"].min()
avg_prob = approved_loans["repayment_probability"].mean()

print(f"Total préstamos aprobados: {len(approved_ids)}")
print(f"Capital total prestado: ${total_allocated:,.2f} USD")
print(f"Presupuesto restante: ${BUDGET_LIMIT - total_allocated:,.2f} USD")
print(f"Probabilidad de pago más baja aprobada: {min_prob:.4f}")
print(f"Probabilidad de pago promedio del portafolio: {avg_prob:.4f}")

Total préstamos aprobados: 687
Capital total prestado: $1,999,800.00 USD
Presupuesto restante: $200.00 USD
Probabilidad de pago más baja aprobada: 0.9105
Probabilidad de pago promedio del portafolio: 0.9546
